## Introduction 

The goal is to predict whether an online visitor will eventually make a purchase based on their browsing behavior (e.g., did they put items in the basket, did they sign in, etc.). This allows e-commerce platforms to target "window shoppers" with specific discounts to convert them into buyers.

## Importing Libraries & Dataset

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

# Styling
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style("whitegrid")

In [28]:
import pandas as pd

df_full = pd.read_csv('User_Behaviour.csv')

# Calculate the fraction needed to get exactly 100,000 rows
frac_needed = 100000 / len(df_full)

#  Take small part of the dataset from the original dataset while maintaining the proportions of 'ordered' 
df = df_full.groupby('ordered', group_keys=False).apply(lambda x: x.sample(frac=frac_needed, random_state=42))
df.reset_index(drop=True, inplace=True)

# 4. Verify the proportions
print("Original Proportions:\n", df_full['ordered'].value_counts(normalize=True))
print("\nNew Sample Proportions:\n", df['ordered'].value_counts(normalize=True))
print("\nNew Sample Total Rows:", len(df))

Original Proportions:
 ordered
0    0.958074
1    0.041926
Name: proportion, dtype: float64

New Sample Proportions:
 ordered
0    0.95807
1    0.04193
Name: proportion, dtype: float64

New Sample Total Rows: 100000


C:\Users\Harshad koli\AppData\Local\Temp\ipykernel_28444\1844155512.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df_full.groupby('ordered', group_keys=False).apply(lambda x: x.sample(frac=frac_needed, random_state=42))


## Data Exploration

In [29]:
df.head(5)

,UserID,basket_icon_click,basket_add_list,basket_add_detail,sort_by,image_picker,account_page_click,promo_banner_click,detail_wishlist_add,list_size_dropdown,...,saw_sizecharts,saw_delivery,saw_account_upgrade,saw_homepage,device_mobile,device_computer,device_tablet,returning_user,loc_uk,ordered
0,67a6-636959c7-67a6-4671-9c21-551077,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,1,0,1,1,0
1,bc88-d06bc27a-bc88-41eb-aa96-845744,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,1,1,0
2,d774-a722d687-d774-4318-a737-728123,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,1,0
3,c696-d0ca5755-c696-49d7-9569-915072,0,0,0,0,0,0,0,0,0,...,0,0,0,1,1,0,0,0,1,0
4,2812-4799aca7-2812-465a-a356-421806,1,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,1,1,0


In [30]:
print("\n--- Statistical Overview ---")
df.describe()


--- Statistical Overview ---


,basket_icon_click,basket_add_list,basket_add_detail,sort_by,image_picker,account_page_click,promo_banner_click,detail_wishlist_add,list_size_dropdown,closed_minibasket_click,...,saw_sizecharts,saw_delivery,saw_account_upgrade,saw_homepage,device_mobile,device_computer,device_tablet,returning_user,loc_uk,ordered
count,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,...,100000.000000,100000.000000,100000.000000,100000.000000,100000.00000,100000.000000,100000.000000,100000.000000,100000.000000,100000.00000
mean,0.099630,0.073920,0.114380,0.037340,0.026890,0.003480,0.016460,0.003390,0.231070,0.017700,...,0.000480,0.005590,0.001160,0.291860,0.67953,0.195620,0.128200,0.537960,0.931920,0.04193
std,0.299508,0.261642,0.318274,0.189595,0.161763,0.058889,0.127237,0.058125,0.421519,0.131859,...,0.021904,0.074557,0.034039,0.454621,0.46666,0.396679,0.334314,0.498559,0.251884,0.20043
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,1.000000,0.00000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,1.00000,0.000000,0.000000,1.000000,1.000000,0.00000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,1.000000,1.00000,0.000000,0.000000,1.000000,1.000000,0.00000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,1.00000


In [31]:
print("--- Columns and Data Types ---") 
df.info()

--- Columns and Data Types ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 25 columns):
 #   Column                   Non-Null Count   Dtype 
---  ------                   --------------   ----- 
 0   UserID                   100000 non-null  object
 1   basket_icon_click        100000 non-null  int64 
 2   basket_add_list          100000 non-null  int64 
 3   basket_add_detail        100000 non-null  int64 
 4   sort_by                  100000 non-null  int64 
 5   image_picker             100000 non-null  int64 
 6   account_page_click       100000 non-null  int64 
 7   promo_banner_click       100000 non-null  int64 
 8   detail_wishlist_add      100000 non-null  int64 
 9   list_size_dropdown       100000 non-null  int64 
 10  closed_minibasket_click  100000 non-null  int64 
 11  checked_delivery_detail  100000 non-null  int64 
 12  checked_returns_detail   100000 non-null  int64 
 13  sign_in                  100000 non-null  in

In [32]:
print("\n--- Missing Values Count ---")
print(df.isnull().sum())


--- Missing Values Count ---
UserID                     0
basket_icon_click          0
basket_add_list            0
basket_add_detail          0
sort_by                    0
image_picker               0
account_page_click         0
promo_banner_click         0
detail_wishlist_add        0
list_size_dropdown         0
closed_minibasket_click    0
checked_delivery_detail    0
checked_returns_detail     0
sign_in                    0
saw_checkout               0
saw_sizecharts             0
saw_delivery               0
saw_account_upgrade        0
saw_homepage               0
device_mobile              0
device_computer            0
device_tablet              0
returning_user             0
loc_uk                     0
ordered                    0
dtype: int64


In [34]:
print("\n--- Target Variable Count (Ordered) ---")
print(df['ordered'].value_counts())


--- Target Variable Count (Ordered) ---
ordered
0    95807
1     4193
Name: count, dtype: int64


The ratio of ordered and not ordered is not good for model traning, will have to perform data modeling so model can be trained properly

ordered
0    151655
Name: count, dtype: int64
Empty DataFrame
Columns: [UserID, basket_icon_click, basket_add_list, basket_add_detail, sort_by, image_picker, account_page_click, promo_banner_click, detail_wishlist_add, list_size_dropdown, closed_minibasket_click, checked_delivery_detail, checked_returns_detail, sign_in, saw_checkout, saw_sizecharts, saw_delivery, saw_account_upgrade, saw_homepage, device_mobile, device_computer, device_tablet, returning_user, loc_uk, ordered]
Index: []

[0 rows x 25 columns]
